# ACL Tear Detection — V6.1 Balanced Regularization

**V6 was too aggressive** — overfitting was fixed but accuracy dropped to ~72%.
V6.1 dials back regularization while keeping the key fix (source+class balanced sampling).

**V6 → V6.1 adjustments:**

| Setting | V5 | V6 (too much) | V6.1 (compromise) |
|---------|----|---------------|-------------------|
| Backbone frozen | 50% | 80% | **70%** |
| Classifier hidden | 256 | 128 | **192** |
| Dropout | 0.4/0.3 | 0.5/0.4 | **0.45/0.35** |
| Weight decay | 0.01 | 0.05 | **0.02** |
| Label smoothing | — | 0.1 | **0.1** (keep) |
| Sampler | class-only | source+class | **source+class** (keep) |
| Augmentation | basic | +blur/erase | **+blur/erase** (keep) |
| Patience | 10 | 5 | **7** |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# Configuration
# ============================================================
DATA_DIR = '/content/drive/MyDrive/dataset/combined'

# Training settings
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = 2  # 0: Normal, 1: Tear (Partial + Complete)
RANDOM_SEED = 42

# Source-aware slice ranges
SLICE_RANGE_PRIYANK = (33, 43)   # Slices 33–42 (inclusive) for Priyank Saxena
SLICE_RANGE_MRI     = (12, 22)   # Slices 12–21 (inclusive) for Kaggle MRI

# CLAHE settings
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID  = (8, 8)

# Percentile normalization
PERCENTILE_LOW  = 2
PERCENTILE_HIGH = 98

# ---- V6.1 Balanced Regularization Settings ----
FREEZE_FRACTION = 0.70       # 70% frozen (V6=80%, V5=50%) — compromise
CLASSIFIER_HIDDEN = 192      # Medium head (V6=128, V5=256) — compromise
DROPOUT_1 = 0.45             # Moderate dropout (V6=0.5, V5=0.4)
DROPOUT_2 = 0.35             # Moderate dropout (V6=0.4, V5=0.3)
LABEL_SMOOTHING = 0.1        # Keep from V6 — prevents overconfident predictions
WEIGHT_DECAY = 0.02          # Moderate (V6=0.05, V5=0.01) — compromise
PATIENCE = 7                 # More room to learn (V6=5, V5=10) — compromise
N_FOLDS = 5                  # K-Fold CV folds

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import cv2
import copy
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# Load and Explore Metadata
# ============================================================
data_path = Path(DATA_DIR)
metadata = pd.read_csv(data_path / 'metadata.csv')

print(f"Total patients: {len(metadata)}")
print(f"\nOriginal label distribution:")
print(metadata['label_name'].value_counts())

metadata['label_binary'] = (metadata['label'] > 0).astype(int)
metadata['label_name_binary'] = metadata['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"\nBinary label distribution:")
print(metadata['label_name_binary'].value_counts())

class_counts_df = metadata['label_binary'].value_counts().sort_index()
imbalance_ratio = class_counts_df.max() / class_counts_df.min()
print(f"\nClass imbalance ratio: {imbalance_ratio:.1f}x")

if 'source' in metadata.columns:
    print(f"\nSource × Label crosstab:")
    print(pd.crosstab(metadata['source'], metadata['label_name_binary'], margins=True))

In [ ]:
# ============================================================
# Preprocessing Functions
# ============================================================

def apply_clahe(img, clip_limit=CLAHE_CLIP_LIMIT, tile_grid=CLAHE_TILE_GRID):
    img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(img_uint8).astype(np.float32) / 255.0

def percentile_normalize(img, low=PERCENTILE_LOW, high=PERCENTILE_HIGH):
    p_low, p_high = np.percentile(img, [low, high])
    img = np.clip(img, p_low, p_high)
    denom = p_high - p_low
    if denom > 0:
        img = (img - p_low) / denom
    return img

In [ ]:
# ============================================================
# Dataset Class
# ============================================================

class ACLSliceDataset(Dataset):
    def __init__(self, df, data_dir, transform=None,
                 slice_range_priyank=SLICE_RANGE_PRIYANK,
                 slice_range_mri=SLICE_RANGE_MRI,
                 use_clahe=True):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.slice_range_priyank = slice_range_priyank
        self.slice_range_mri = slice_range_mri
        self.use_clahe = use_clahe

        self.samples = []
        skipped = 0
        for idx, row in self.df.iterrows():
            num_slices = int(row['num_slices'])
            source = row.get('source', 'unknown')

            if source == 'Priyank_Saxena':
                start, end = self.slice_range_priyank
            else:
                start, end = self.slice_range_mri

            start = min(start, num_slices)
            end = min(end, num_slices)

            if start >= end:
                skipped += 1
                continue

            for slice_idx in range(start, end):
                self.samples.append((idx, slice_idx, int(row['label_binary']), source))

        if skipped > 0:
            print(f"  ⚠️ Skipped {skipped} patients (slice range out of bounds)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        patient_idx, slice_idx, label, source = self.samples[idx]
        row = self.df.iloc[patient_idx]

        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']
        slice_idx = min(slice_idx, volume.shape[0] - 1)

        img = volume[slice_idx].astype(np.float32) / 255.0
        img = percentile_normalize(img)

        if self.use_clahe:
            img = apply_clahe(img)

        img_mean = img.mean()
        img_std = img.std() + 1e-8
        img = (img - img_mean) / img_std

        img = np.stack([img, img, img], axis=0)
        img = torch.from_numpy(img)

        if self.transform:
            img = self.transform(img)

        return img, label, patient_idx

    def get_labels(self):
        return [s[2] for s in self.samples]

    def get_sources(self):
        return [s[3] for s in self.samples]

In [ ]:
# ============================================================
# Source + Class Balanced Sampler (KEY FIX — kept from V6)
# ============================================================

def make_source_class_balanced_sampler(dataset):
    labels = dataset.get_labels()
    sources = dataset.get_sources()

    group_counts = Counter()
    for label, source in zip(labels, sources):
        group_counts[(source, label)] += 1

    print("  Sample counts per (source, class) group:")
    for key, count in sorted(group_counts.items()):
        print(f"    {key[0]:20s} - {'Normal' if key[1]==0 else 'Tear':6s}: {count}")

    sample_weights = []
    for label, source in zip(labels, sources):
        group_count = group_counts[(source, label)]
        sample_weights.append(1.0 / group_count)

    sampler = WeightedRandomSampler(
        sample_weights, len(sample_weights), replacement=True
    )
    return sampler

In [ ]:
# ============================================================
# Data Augmentation (kept from V6)
# ============================================================

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_transform = None

In [ ]:
# ============================================================
# Model Factory — V6.1 compromise capacity
# ============================================================

def create_model(freeze_fraction=FREEZE_FRACTION,
                 hidden_size=CLASSIFIER_HIDDEN,
                 dropout1=DROPOUT_1,
                 dropout2=DROPOUT_2):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')

    all_params = list(model.features.parameters())
    freeze_until = int(len(all_params) * freeze_fraction)
    for i, param in enumerate(all_params):
        param.requires_grad = (i >= freeze_until)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout1),
        nn.Linear(in_features, hidden_size),
        nn.ReLU(),
        nn.Dropout(p=dropout2),
        nn.Linear(hidden_size, NUM_CLASSES)
    )

    model = model.to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Backbone: {freeze_fraction*100:.0f}% frozen")
    print(f"  Classifier: {hidden_size} hidden, dropout={dropout1}/{dropout2}")
    print(f"  Params: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)")
    return model

In [ ]:
# ============================================================
# Training & Validation Functions
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels, _ in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_patient_indices = []

    with torch.no_grad():
        for images, labels, patient_idx in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_patient_indices.extend(patient_idx.numpy())

    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels, all_patient_indices

In [ ]:
# ============================================================
# Patient-Level Majority Voting Helper
# ============================================================

def patient_level_metrics(preds, labels, patient_indices, dataset_df=None,
                          label_names=['Normal', 'Tear'], verbose=True):
    patient_preds_dict = {}
    patient_labels_dict = {}
    patient_source_dict = {}

    for pred, label, pidx in zip(preds, labels, patient_indices):
        pidx = int(pidx)
        if pidx not in patient_preds_dict:
            patient_preds_dict[pidx] = []
            patient_labels_dict[pidx] = label
            if dataset_df is not None:
                patient_source_dict[pidx] = dataset_df.iloc[pidx].get('source', 'unknown')
        patient_preds_dict[pidx].append(pred)

    patient_final_preds = []
    patient_final_labels = []
    patient_sources = []

    for pidx in sorted(patient_preds_dict.keys()):
        final_pred = Counter(patient_preds_dict[pidx]).most_common(1)[0][0]
        patient_final_preds.append(final_pred)
        patient_final_labels.append(patient_labels_dict[pidx])
        if pidx in patient_source_dict:
            patient_sources.append(patient_source_dict[pidx])

    patient_acc = np.mean(
        np.array(patient_final_preds) == np.array(patient_final_labels)
    ) * 100

    if verbose:
        print(f"\nPatient-level Accuracy: {patient_acc:.1f}% ({len(patient_final_preds)} patients)")
        print(classification_report(
            patient_final_labels, patient_final_preds,
            target_names=label_names, digits=3, zero_division=0
        ))

        if patient_sources:
            for source_name in sorted(set(patient_sources)):
                s_preds = [p for p, s in zip(patient_final_preds, patient_sources) if s == source_name]
                s_labels = [l for l, s in zip(patient_final_labels, patient_sources) if s == source_name]
                if s_preds:
                    s_acc = np.mean(np.array(s_preds) == np.array(s_labels)) * 100
                    print(f"  {source_name}: {s_acc:.1f}% ({len(s_preds)} patients)")
                    print(classification_report(
                        s_labels, s_preds,
                        target_names=label_names, digits=3, zero_division=0
                    ))

    return patient_acc, patient_final_preds, patient_final_labels, patient_sources

---
## Part A: Single-Split Training

In [ ]:
# ============================================================
# Train/Val/Test Split
# ============================================================
train_df, temp_df = train_test_split(
    metadata, test_size=0.3, stratify=metadata['label_binary'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_binary'], random_state=RANDOM_SEED
)

print(f"Train: {len(train_df)} patients")
print(f"Val:   {len(val_df)} patients")
print(f"Test:  {len(test_df)} patients")

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n{name}:")
    print(pd.crosstab(df['source'], df['label_name_binary']))

In [ ]:
# ============================================================
# Create Datasets
# ============================================================
print("Creating datasets with source-aware slice selection...")
print(f"  Priyank Saxena: slices {SLICE_RANGE_PRIYANK[0]}–{SLICE_RANGE_PRIYANK[1]-1}")
print(f"  Kaggle MRI:     slices {SLICE_RANGE_MRI[0]}–{SLICE_RANGE_MRI[1]-1}")
print()

print("Train:")
train_dataset = ACLSliceDataset(train_df, DATA_DIR, train_transform)
print("Val:")
val_dataset = ACLSliceDataset(val_df, DATA_DIR, val_transform)
print("Test:")
test_dataset = ACLSliceDataset(test_df, DATA_DIR, val_transform)

print(f"\nTrain samples (slices): {len(train_dataset)}")
print(f"Val samples (slices):   {len(val_dataset)}")
print(f"Test samples (slices):  {len(test_dataset)}")

In [ ]:
# ============================================================
# Source + Class Balanced Sampler & DataLoaders
# ============================================================
print("\nCreating source + class balanced sampler...")
train_sampler = make_source_class_balanced_sampler(train_dataset)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# ============================================================
# Create Model, Loss, Optimizer
# ============================================================
print("\nV6.1 Model:")
model = create_model()

train_labels = train_dataset.get_labels()
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=LABEL_SMOOTHING
)

optimizer = optim.AdamW(
    model.parameters(), lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

print(f"\nV6.1 settings:")
print(f"  Freeze:          {FREEZE_FRACTION*100:.0f}%")
print(f"  Classifier:      {CLASSIFIER_HIDDEN} hidden")
print(f"  Dropout:         {DROPOUT_1}/{DROPOUT_2}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Weight decay:    {WEIGHT_DECAY}")
print(f"  Patience:        {PATIENCE}")

In [ ]:
# ============================================================
# Training Loop
# ============================================================
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
patience_counter = 0

SAVE_PATH = '/content/drive/MyDrive/dataset/best_acl_model_v6_1.pth'

print(f"Training for up to {NUM_EPOCHS} epochs (patience={PATIENCE})...\n")

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_preds, val_labels, val_pidx = validate(
        model, val_loader, criterion, device
    )
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    gap = train_acc - val_acc
    gap_warning = " ⚠️ OVERFIT" if gap > 10 else ""

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train: loss={train_loss:.4f}  acc={train_acc:.2f}%")
    print(f"  Val:   loss={val_loss:.4f}  acc={val_acc:.2f}%  (gap={gap:.1f}%){gap_warning}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  → Saved best model (val_acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  → No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

In [ ]:
# ============================================================
# Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

gaps = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
axes[2].plot(gaps, label='Train-Val Gap', color='red')
axes[2].axhline(y=10, color='orange', linestyle='--', label='Warning threshold (10%)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy Gap (%)')
axes[2].set_title('Overfitting Gap (lower is better)')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/training_history_v6_1.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Evaluate on Test Set
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))

test_loss, test_acc, test_preds, test_labels, test_pidx = validate(
    model, test_loader, criterion, device
)

print("=" * 60)
print("TEST RESULTS — V6.1 (Single Split)")
print("=" * 60)
print(f"Slice-level Accuracy: {test_acc:.2f}%")

label_names = ['Normal', 'Tear']
print("\nSlice-level Classification Report:")
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Slice Level (V6.1)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v6_1_slice.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Patient-Level Results
# ============================================================
print("=" * 60)
print("PATIENT-LEVEL RESULTS (Majority Voting)")
print("=" * 60)

patient_acc, p_preds, p_labels, p_sources = patient_level_metrics(
    test_preds, test_labels, test_pidx, test_dataset.df
)

cm_patient = confusion_matrix(p_labels, p_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_patient, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Patient Level (V6.1)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v6_1_patient.png', dpi=150)
plt.show()

---
## Part B: 5-Fold Stratified Cross-Validation

In [ ]:
# ============================================================
# Stratified K-Fold Cross-Validation
# ============================================================
print(f"\n{'='*60}")
print(f"{N_FOLDS}-FOLD STRATIFIED CROSS-VALIDATION")
print(f"{'='*60}\n")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_results = []
all_fold_preds = []
all_fold_labels = []
all_fold_sources = []

for fold, (train_idx, val_idx) in enumerate(skf.split(metadata, metadata['label_binary'])):
    print(f"\n{'─'*50}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'─'*50}")

    fold_train_df = metadata.iloc[train_idx]
    fold_val_df = metadata.iloc[val_idx]

    print(f"  Train: {len(fold_train_df)} patients")
    print(f"  Val:   {len(fold_val_df)} patients")
    print(f"  Val source distribution: {fold_val_df['source'].value_counts().to_dict()}")

    fold_train_dataset = ACLSliceDataset(fold_train_df, DATA_DIR, train_transform)
    fold_val_dataset = ACLSliceDataset(fold_val_df, DATA_DIR, val_transform)

    fold_sampler = make_source_class_balanced_sampler(fold_train_dataset)

    fold_train_loader = DataLoader(
        fold_train_dataset, batch_size=BATCH_SIZE, sampler=fold_sampler, num_workers=2
    )
    fold_val_loader = DataLoader(
        fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    print("  Creating model...")
    fold_model = create_model()

    fold_train_labels = fold_train_dataset.get_labels()
    fold_class_counts = np.bincount(fold_train_labels)
    fold_class_weights = 1.0 / fold_class_counts
    fold_cw_tensor = torch.tensor(fold_class_weights, dtype=torch.float32).to(device)

    fold_criterion = nn.CrossEntropyLoss(
        weight=fold_cw_tensor, label_smoothing=LABEL_SMOOTHING
    )
    fold_optimizer = optim.AdamW(
        fold_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    fold_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        fold_optimizer, T_0=10, T_mult=2
    )

    best_fold_acc = 0
    fold_patience = 0
    best_fold_state = None

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_epoch(
            fold_model, fold_train_loader, fold_criterion, fold_optimizer, device
        )
        val_loss, val_acc, _, _, _ = validate(
            fold_model, fold_val_loader, fold_criterion, device
        )
        fold_scheduler.step()

        if val_acc > best_fold_acc:
            best_fold_acc = val_acc
            best_fold_state = copy.deepcopy(fold_model.state_dict())
            fold_patience = 0
        else:
            fold_patience += 1
            if fold_patience >= PATIENCE:
                print(f"  Early stop at epoch {epoch+1} (best val_acc: {best_fold_acc:.2f}%)")
                break

    fold_model.load_state_dict(best_fold_state)
    _, fold_acc, fold_preds, fold_labels, fold_pidx = validate(
        fold_model, fold_val_loader, fold_criterion, device
    )

    p_acc, fp_preds, fp_labels, fp_sources = patient_level_metrics(
        fold_preds, fold_labels, fold_pidx, fold_val_dataset.df, verbose=True
    )

    fold_results.append({
        'fold': fold + 1,
        'slice_acc': fold_acc,
        'patient_acc': p_acc,
        'best_epoch': epoch + 1 - PATIENCE if fold_patience >= PATIENCE else epoch + 1,
    })

    all_fold_preds.extend(fp_preds)
    all_fold_labels.extend(fp_labels)
    all_fold_sources.extend(fp_sources)

    del fold_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# K-Fold Summary
# ============================================================
print(f"\n{'='*60}")
print(f"K-FOLD CROSS-VALIDATION SUMMARY")
print(f"{'='*60}")

results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))

print(f"\n{'─'*40}")
print(f"Mean Slice-Level Accuracy:   {results_df['slice_acc'].mean():.2f}% ± {results_df['slice_acc'].std():.2f}%")
print(f"Mean Patient-Level Accuracy: {results_df['patient_acc'].mean():.2f}% ± {results_df['patient_acc'].std():.2f}%")

print(f"\n{'─'*40}")
print("Overall Patient-Level Report (all folds combined):")
print(classification_report(
    all_fold_labels, all_fold_preds,
    target_names=label_names, digits=3, zero_division=0
))

print("Per-Source Overall (all folds combined):")
for source_name in sorted(set(all_fold_sources)):
    s_preds = [p for p, s in zip(all_fold_preds, all_fold_sources) if s == source_name]
    s_labels = [l for l, s in zip(all_fold_labels, all_fold_sources) if s == source_name]
    if s_preds:
        s_acc = np.mean(np.array(s_preds) == np.array(s_labels)) * 100
        print(f"\n  {source_name}: {s_acc:.1f}% ({len(s_preds)} patients)")
        print(classification_report(
            s_labels, s_preds,
            target_names=label_names, digits=3, zero_division=0
        ))

In [ ]:
# ============================================================
# K-Fold Visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = results_df['fold']
axes[0].bar(x - 0.15, results_df['slice_acc'], width=0.3, label='Slice-Level', color='steelblue')
axes[0].bar(x + 0.15, results_df['patient_acc'], width=0.3, label='Patient-Level', color='coral')
axes[0].axhline(y=results_df['patient_acc'].mean(), color='coral', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy per Fold (V6.1)')
axes[0].legend()
axes[0].set_xticks(x)
axes[0].grid(axis='y')

cm_all = confusion_matrix(all_fold_labels, all_fold_preds)
sns.heatmap(cm_all, annot=True, fmt='d', cmap='Purples',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title(f'Confusion Matrix — {N_FOLDS}-Fold CV (Patient Level, V6.1)')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/kfold_results_v6_1.png', dpi=150)
plt.show()